In [1]:
from google.colab import drive
# Mount Google Drive/
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import zipfile
import tarfile
import shutil

# List of zip files
zip_files = [
   "/content/drive/MyDrive/annotations_v2.zip",
   "/content/drive/MyDrive/data by hand.zip",
   "/content/drive/MyDrive/train_images.zip",
   "/content/drive/MyDrive/validation_images.zip",
   "/content/drive/MyDrive/dev_gold_labels.zip",
   "/content/drive/MyDrive/dev_images.zip"



]

# Iterate through each zip file
for zip_file in zip_files:
    # Extract the filename without extension
    file_name = os.path.splitext(os.path.basename(zip_file))[0]

    # Create a directory for each file
    extract_dir = os.path.join("/content", file_name)
    os.makedirs(extract_dir, exist_ok=True)

    try:
        # Check if the file is a zip archive
        if zipfile.is_zipfile(zip_file):
            with zipfile.ZipFile(zip_file, 'r') as zip_ref:
                zip_ref.extractall(extract_dir)
                print(f"Extracted {zip_file} to {extract_dir}")
        # Check if the file is a tar archive
        elif tarfile.is_tarfile(zip_file):
            with tarfile.open(zip_file, 'r') as tar_ref:
                tar_ref.extractall(extract_dir)
                print(f"Extracted {zip_file} to {extract_dir}")
        else:
            print(f"Skipping {zip_file} as it is not a zip or tar archive")
    except Exception as e:
        print(f"Error extracting {zip_file}: {e}")

Extracted /content/drive/MyDrive/annotations_v2.zip to /content/annotations_v2
Extracted /content/drive/MyDrive/data by hand.zip to /content/data by hand
Extracted /content/drive/MyDrive/train_images.zip to /content/train_images
Extracted /content/drive/MyDrive/validation_images.zip to /content/validation_images
Extracted /content/drive/MyDrive/dev_gold_labels.zip to /content/dev_gold_labels
Extracted /content/drive/MyDrive/dev_images.zip to /content/dev_images


In [4]:
import json
import pandas as pd

# Load JSON file
with open('/content/drive/MyDrive/dataset_with_rationales_subtask2a_final (1).json', 'r') as f:
    data = json.load(f)

# Convert to DataFrame
df = pd.DataFrame(data)



# Display as table
display(df.head())



,id,text,image,labels,link,objects,caption,rationale
0,63292,This is why we're free\n\nThis is why we're sa...,prop_meme_556.png,"[Causal Oversimplification, Transfer, Flag-wav...",https://www.facebook.com/SilentmajorityDJT/pho...,"White text ""This is why we're free"", dark back...",This meme uses a split image to convey a clear...,The meme effectively combines text and imagery...
1,65635,THIS IS WHY YOU NEED\n\nA SHARPIE WITH YOU AT ...,prop_meme_4839.png,"[Transfer, Black-and-white Fallacy/Dictatorshi...",https://www.facebook.com/photo/?fbid=402355213...,Visible elements include: a weathered concrete...,This meme humorously advocates for carrying a ...,The text-image interaction cleverly juxtaposes...
2,67927,GOOD NEWS!\n\nNAZANIN ZAGHARI-RATCLIFFE AND AN...,prop_meme_7653.png,"[Loaded Language, Glittering generalities (Vir...",https://www.facebook.com/amnesty/photos/531198...,"Teal/light blue gradient background, black tex...",This meme powerfully conveys the celebratory n...,The text and imagery in the meme work in harmo...
3,68031,PAING PHYO MIN IS FREE!,prop_meme_7826.png,[Glittering generalities (Virtue)],https://www.facebook.com/amnesty/photos/427419...,"A young man (Paing Phyo Min), short dark hair,...",This meme celebrates a significant victory for...,The interplay between the bold text and Paing ...
4,77490,Move your ships away!\n\noooook\n\nMove your s...,prop_meme_18807.png,[Smears],https://www.facebook.com/rightpatriots/photos/...,The image displays a four-panel meme. Visible ...,This meme satirizes the perceived diplomatic s...,The interaction between text and image in this...


In [ ]:
print(f"The dataset has {len(df)} records.")

The dataset has 500 records.


In [ ]:
import pandas as pd
import json

# Use the DataFrame 'df' loaded in the previous cell (QhWBjnkuTheA)
# Rename the column 'gemini_caption' to 'caption'
df.rename(columns={'gemini_caption': 'caption'}, inplace=True)

# Convert DataFrame to JSON format
# Use orient='records' to get a list of dictionaries, which is a common JSON format for tabular data
json_data = df.to_dict(orient='records')

# Define the output path for the JSON file
output_path = '/content/drive/MyDrive/validation_caption.json'

# Save the JSON data to a file
with open(output_path, 'w') as f:
    json.dump(json_data, f, indent=4) # Use indent for pretty printing

print(f"Updated DataFrame saved to: {output_path}")

Updated DataFrame saved to: /content/drive/MyDrive/validation_caption.json


In [ ]:
 {
    "Persuasion": ["Ethos", "Logos", "Pathos"],
    "Ethos": ["Ad Hominem", "Justification"],
    "Logos": ["Distraction", "Simplification"],
    "Pathos": ["Other", "Other"],
    "Ad Hominem": ["Name calling/Labeling", "Doubt", "Smears", "Reductio ad hitlerum", "Whataboutism"],
    "Justification": ["Flag-waving", "Appeal to fear/prejudice", "Bandwagon", "Slogans"],
    "Distraction": ["Misrepresentation of Someone's Position (Straw Man)", "Presenting Irrelevant Data (Red Herring)", "Whataboutism"],
    "Simplification": ["Black-and-white Fallacy/Dictatorship", "Thought-terminating cliché", "Causal Oversimplification"],
    "Other": ["Bandwagon", "Appeal to authority", "Glittering generalities (Virtue)", "Transfer", "Repetition",
             "Obfuscation, Intentional vagueness, Confusion", "Appeal to (Strong) Emotions", "Exaggeration/Minimisation",
             "Loaded Language", "Flag-waving", "Appeal to fear/prejudice", "Transfer"]
}

In [5]:
"""
AdaBoost Implementation for Propaganda Detection
این کد AdaBoost رو به کد پایه شما اضافه می‌کنه
"""

import os
import json
import random
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, precision_recall_curve
from sklearn.metrics.pairwise import cosine_similarity
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from collections import Counter
import copy

from transformers import CLIPModel, CLIPProcessor, RobertaModel, RobertaTokenizer
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION - SAME AS YOUR BASE CODE
# ============================================================================
class CFG:
    # Paths
    train_json = '/content/drive/MyDrive/dataset_with_rationales_subtask2a_final (1).json'
    val_json = '/content/drive/MyDrive/validation_caption.json'
    test_json = '/content/drive/MyDrive/dev_processed.json'

    train_img_dir = '/content/train_images/train_images'
    val_img_dir   = '/content/validation_images/validation_images'
    test_img_dir  = '/content/dev_images/dev_images'

    # Hyperparameters
    seed = 42
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    batch_size = 32
    lr = 2e-3
    epochs = 8  # تعداد epoch برای هر weak learner کمتر می‌شه

    # AdaBoost specific
    n_estimators = 5  # تعداد weak learners
    adaboost_learning_rate = 0.8  # برای update کردن sample weights

    # Caption settings
    use_caption = True
    caption_separator = " [SEP] "

    # Model paths
    checkpoint_dir = './checkpoints_adaboost'
    log_dir = './logs_adaboost'

    # Model names
    clip_model_name = "openai/clip-vit-base-patch32"
    roberta_model_name = "roberta-base"

    # Training improvements
    gradient_clip_norm = 1.0

    # GCN settings
    use_gcn = True
    gcn_hidden_dim = 256
    gcn_layers = 2
    pmi_weight = 0.6
    semantic_weight = 0.4

def set_seed(seed=42):
    """Set random seeds for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ============================================================================
# همون HIERARCHY و DEFINITIONS از کد شما
# ============================================================================
HIERARCHY_GRAPH = {
    "Persuasion": ["Ethos", "Logos", "Pathos"],
    "Ethos": ["Ad Hominem", "Justification"],
    "Logos": ["Distraction", "Simplification"],
    "Pathos": ["Other", "Other"],
    "Ad Hominem": ["Name calling/Labeling", "Doubt", "Smears", "Reductio ad hitlerum", "Whataboutism"],
    "Justification": ["Flag-waving", "Appeal to fear/prejudice", "Bandwagon", "Slogans"],
    "Distraction": ["Misrepresentation of Someone's Position (Straw Man)", "Presenting Irrelevant Data (Red Herring)", "Whataboutism"],
    "Simplification": ["Black-and-white Fallacy/Dictatorship", "Thought-terminating cliché", "Causal Oversimplification"],
    "Other": ["Bandwagon", "Appeal to authority", "Glittering generalities (Virtue)", "Transfer", "Repetition",
             "Obfuscation, Intentional vagueness, Confusion", "Appeal to (Strong) Emotions", "Exaggeration/Minimisation",
             "Loaded Language", "Flag-waving", "Appeal to fear/prejudice", "Transfer"]
}

TECHNIQUE_DEFINITIONS = {
    "Name calling/Labeling": "Giving a person or idea a bad label to make the audience reject them without examining evidence",
    "Repetition": "Repeating the same message over and over again so that the audience will accept it",
    "Slogans": "A brief and striking phrase that contains labeling and stereotyping",
    "Appeal to fear/prejudice": "Seeking to build support by instilling anxiety and panic in the population",
    "Doubt": "Questioning the credibility of someone or something",
    "Exaggeration/Minimisation": "Either representing something in an excessive manner or making something seem less important",
    "Flag-waving": "Playing on strong national feeling to justify or promote an action",
    "Causal Oversimplification": "Assuming a single cause when there are multiple causes behind an issue",
    "Appeal to authority": "Supposing that a claim is true because a valid authority or expert on the issue said it",
    "Black-and-white Fallacy/Dictatorship": "Presenting two alternative options as the only possibilities",
    "Thought-terminating cliché": "Words or phrases that discourage critical thought and useful discussion",
    "Whataboutism": "Discredit an opponent's position by charging them with hypocrisy without refuting their argument",
    "Reductio ad hitlerum": "Comparing something/someone to Hitler or Nazism to make the argument seem invalid",
    "Bandwagon": "Attempting to persuade the target audience to join in and take the course of action because everyone else is doing so",
    "Obfuscation, Intentional vagueness, Confusion": "Using deliberately unclear words to make the message confusing",
    "Loaded Language": "Using specific words and phrases with strong emotional implications to influence the audience",
    "Glittering generalities (Virtue)": "Words associated with highly valued concepts that are used to evoke positive emotional response",
    "Misrepresentation of Someone's Position (Straw Man)": "When an opponent's proposition is substituted with a similar one",
    "Presenting Irrelevant Data (Red Herring)": "Introducing irrelevant material to the argument to distract",
    "Transfer": "Projecting positive or negative qualities of a person, entity, object to another",
    "Appeal to (Strong) Emotions": "Attempting to develop an emotional response instead of a valid or compelling argument",
    "Smears": "A direct attack on the reputation or character of a person or group",
}

# ============================================================================
# استفاده از همون توابع کمکی شما
# ============================================================================

def create_label_mappings():
    """Create label to index mappings and ancestor matrix"""
    all_labels = set(HIERARCHY_GRAPH.keys())
    for children in HIERARCHY_GRAPH.values():
        all_labels.update(children)

    label_to_idx = {label: i for i, label in enumerate(sorted(all_labels))}
    idx_to_label = {i: label for label, i in label_to_idx.items()}
    num_labels = len(label_to_idx)

    ancestors = {label_to_idx['Persuasion']: {label_to_idx['Persuasion']}}

    for parent, children in HIERARCHY_GRAPH.items():
        parent_idx = label_to_idx[parent]
        for child in children:
            child_idx = label_to_idx[child]
            ancestors[child_idx] = ancestors.get(child_idx, set()) | ancestors.get(parent_idx, set()) | {child_idx}

    ancestor_matrix = torch.zeros((num_labels, num_labels), dtype=torch.float32)
    for node, anc_set in ancestors.items():
        ancestor_matrix[node, list(anc_set)] = 1.0

    return label_to_idx, idx_to_label, ancestor_matrix, num_labels

def compute_pmi_matrix(labels_data, label_to_idx, smooth=1e-5):
    """محاسبه PMI matrix - همون کد شما"""
    num_labels = len(label_to_idx)
    co_occurrence = np.zeros((num_labels, num_labels))
    label_counts = np.zeros(num_labels)
    total_samples = len(labels_data)

    for labels in tqdm(labels_data, desc="Computing co-occurrences"):
        label_indices = [label_to_idx[label] for label in labels if label in label_to_idx]
        for idx in label_indices:
            label_counts[idx] += 1
        for i in label_indices:
            for j in label_indices:
                co_occurrence[i, j] += 1

    pmi_matrix = np.zeros((num_labels, num_labels))
    for i in range(num_labels):
        for j in range(num_labels):
            p_i = (label_counts[i] + smooth) / total_samples
            p_j = (label_counts[j] + smooth) / total_samples
            p_ij = (co_occurrence[i, j] + smooth) / total_samples
            pmi = np.log(p_ij / (p_i * p_j))
            pmi_matrix[i, j] = max(0, pmi)

    if pmi_matrix.max() > 0:
        pmi_matrix = pmi_matrix / pmi_matrix.max()

    return pmi_matrix

def compute_semantic_similarity(label_to_idx, idx_to_label, roberta_model, roberta_tokenizer, device):
    """محاسبه semantic similarity - همون کد شما"""
    num_labels = len(label_to_idx)
    similarity_matrix = np.zeros((num_labels, num_labels))
    embeddings = []

    with torch.no_grad():
        for i in tqdm(range(num_labels), desc="Extracting embeddings"):
            label_name = idx_to_label[i]
            definition = TECHNIQUE_DEFINITIONS.get(label_name, label_name)

            encoded = roberta_tokenizer(
                definition, padding='max_length', truncation=True,
                max_length=128, return_tensors='pt'
            )

            input_ids = encoded['input_ids'].to(device)
            attention_mask = encoded['attention_mask'].to(device)
            outputs = roberta_model(input_ids=input_ids, attention_mask=attention_mask)
            embedding = outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()
            embeddings.append(embedding)

    embeddings = np.array(embeddings)
    similarity_matrix = cosine_similarity(embeddings)
    similarity_matrix = (similarity_matrix + 1) / 2
    return similarity_matrix

def build_label_graph(train_df, label_to_idx, idx_to_label, roberta_model, roberta_tokenizer, device,
                      pmi_weight=0.6, semantic_weight=0.4):
    """ساخت label graph - همون کد شما"""
    print("\n" + "="*60)
    print("BUILDING LABEL GRAPH FOR GCN")
    print("="*60)

    labels_data = train_df['labels'].tolist()
    pmi_matrix = compute_pmi_matrix(labels_data, label_to_idx)
    semantic_matrix = compute_semantic_similarity(label_to_idx, idx_to_label,
                                                   roberta_model, roberta_tokenizer, device)

    adjacency_matrix = pmi_weight * pmi_matrix + semantic_weight * semantic_matrix
    np.fill_diagonal(adjacency_matrix, 1.0)

    degree = adjacency_matrix.sum(axis=1)
    degree_inv_sqrt = np.power(degree, -0.5)
    degree_inv_sqrt[np.isinf(degree_inv_sqrt)] = 0.0
    degree_matrix_inv_sqrt = np.diag(degree_inv_sqrt)
    adjacency_matrix_norm = degree_matrix_inv_sqrt @ adjacency_matrix @ degree_matrix_inv_sqrt

    print(f"✓ Label graph built: {adjacency_matrix_norm.shape}")
    return torch.FloatTensor(adjacency_matrix_norm)

# ============================================================================
# همون کلاس‌های کمکی شما (FocalLoss, GCN, MLP, etc.)
# ============================================================================

class FocalLoss(nn.Module):
    """Focal Loss"""
    def __init__(self, alpha=1.0, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

class GraphConvolutionLayer(nn.Module):
    """Graph Convolution Layer"""
    def __init__(self, in_features, out_features, bias=True):
        super(GraphConvolutionLayer, self).__init__()
        self.weight = nn.Parameter(torch.FloatTensor(in_features, out_features))
        if bias:
            self.bias = nn.Parameter(torch.FloatTensor(out_features))
        else:
            self.register_parameter('bias', None)
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.weight)
        if self.bias is not None:
            nn.init.zeros_(self.bias)

    def forward(self, input_features, adjacency_matrix):
        support = torch.matmul(input_features, self.weight)
        output = torch.einsum('ij,bjf->bif', adjacency_matrix, support)
        if self.bias is not None:
            output = output + self.bias
        return output

class LabelGCN(nn.Module):
    """GCN برای label refinement - همون کد شما"""
    def __init__(self, num_labels, hidden_dim=256, num_layers=2, dropout=0.3):
        super(LabelGCN, self).__init__()
        self.num_labels = num_labels
        self.num_layers = num_layers
        self.gcn_layers = nn.ModuleList()

        self.gcn_layers.append(GraphConvolutionLayer(1, hidden_dim))
        for _ in range(num_layers - 2):
            self.gcn_layers.append(GraphConvolutionLayer(hidden_dim, hidden_dim))
        if num_layers > 1:
            self.gcn_layers.append(GraphConvolutionLayer(hidden_dim, 1))

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.batch_norm = nn.ModuleList([nn.BatchNorm1d(hidden_dim) for _ in range(num_layers - 1)])
        self.residual_weight = nn.Parameter(torch.FloatTensor([0.5]))

    def forward(self, predictions, adjacency_matrix):
        x = predictions.unsqueeze(-1)

        for i, gcn_layer in enumerate(self.gcn_layers):
            x = gcn_layer(x, adjacency_matrix)
            if i < len(self.gcn_layers) - 1:
                x = x.transpose(1, 2)
                x = self.batch_norm[i](x)
                x = x.transpose(1, 2)
                x = self.relu(x)
                x = self.dropout(x)

        refined = x.squeeze(-1)
        alpha = torch.sigmoid(self.residual_weight)
        output = alpha * predictions + (1 - alpha) * refined
        return output

class ImprovedMultiHeadMLP(nn.Module):
    """همون Multi-Head MLP شما با GCN"""
    def __init__(self, input_dim=1792, num_labels=22, use_gcn=True, gcn_hidden_dim=256, gcn_layers=2):
        super(ImprovedMultiHeadMLP, self).__init__()
        self.use_gcn = use_gcn

        # Head 1 layers
        self.layer1 = nn.Linear(input_dim, 768)
        self.layer1_bn = nn.BatchNorm1d(768)
        self.layer2 = nn.Linear(768, 512)
        self.layer2_bn = nn.BatchNorm1d(512)
        self.head1 = nn.Linear(512, 3)
        self.head1_reducer = nn.Linear(512, 64)
        self.head1_to_head2_residual = nn.Linear(512, 128)

        # Head 2 layers
        self.layer3 = nn.Linear(512, 256)
        self.layer3_bn = nn.BatchNorm1d(256)
        self.layer4 = nn.Linear(256, 128)
        self.layer4_bn = nn.BatchNorm1d(128)
        self.head2 = nn.Linear(128, 5)
        self.head2_reducer = nn.Linear(128, 64)
        self.head2_to_final_residual = nn.Linear(128, 64)

        # Head 3 (final) layers
        self.final_layer1 = nn.Linear(128, 96)
        self.final_layer1_bn = nn.BatchNorm1d(96)
        self.final_layer2 = nn.Linear(96, 64)
        self.final_layer2_bn = nn.BatchNorm1d(64)
        self.final_residual = nn.Linear(128, 64)
        self.final_head = nn.Linear(64, num_labels)

        # GCN
        if self.use_gcn:
            self.gcn = LabelGCN(num_labels, gcn_hidden_dim, gcn_layers, dropout=0.3)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)

    def forward(self, x, adjacency_matrix=None):
        # Head 1
        x1 = self.relu(self.layer1_bn(self.layer1(x)))
        x1 = self.dropout(x1)
        x1 = self.relu(self.layer2_bn(self.layer2(x1)))
        x1 = self.dropout(x1)
        output1 = self.head1(x1)
        head1_features = self.relu(self.head1_reducer(x1))
        head1_features = self.dropout(head1_features)

        # Head 2
        x2 = self.relu(self.layer3_bn(self.layer3(x1)))
        x2 = self.dropout(x2)
        x2 = self.relu(self.layer4_bn(self.layer4(x2)))
        x2 = self.dropout(x2)
        x2 = x2 + self.head1_to_head2_residual(x1)
        output2 = self.head2(x2)
        head2_features = self.relu(self.head2_reducer(x2))
        head2_features = self.dropout(head2_features)

        # Head 3
        combined = torch.cat([head1_features, head2_features], dim=1)
        x3 = self.relu(self.final_layer1_bn(self.final_layer1(combined)))
        x3 = self.dropout(x3)
        x3 = self.relu(self.final_layer2_bn(self.final_layer2(x3)))
        x3 = self.dropout(x3)
        x3 = x3 + self.final_residual(combined)
        output_final = self.final_head(x3)

        # GCN refinement
        if self.use_gcn and adjacency_matrix is not None:
            output_final = self.gcn(output_final, adjacency_matrix)

        return output1, output2, output_final

class MemeDataset(Dataset):
    """همون Dataset شما"""
    def __init__(self, df, img_dir, processor, label_to_idx, ancestor_matrix,
                 is_test=False, use_caption=True, caption_separator=" [SEP] "):
        self.df = df.reset_index(drop=True)
        self.img_dir = Path(img_dir)
        self.processor = processor
        self.label_to_idx = label_to_idx
        self.ancestor_matrix = ancestor_matrix
        self.num_labels = len(label_to_idx)
        self.is_test = is_test
        self.use_caption = use_caption
        self.caption_separator = caption_separator

    def __len__(self):
        return len(self.df)

    def _encode_labels(self, label_list):
        if not label_list:
            return torch.zeros(self.num_labels)

        label_indices = [self.label_to_idx[label] for label in label_list if label in self.label_to_idx]
        expanded_indices = set()
        for idx in label_indices:
            ancestors = torch.where(self.ancestor_matrix[idx] == 1)[0].tolist()
            expanded_indices.update(ancestors)

        y = torch.zeros(self.num_labels)
        y[list(expanded_indices)] = 1.0
        return y

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = row['text'] if 'text' in row and pd.notna(row['text']) else ""

        if self.use_caption and 'caption' in row and pd.notna(row['caption']):
            caption = row['caption']
            combined_text = f"{text}{self.caption_separator}{caption}"
        else:
            combined_text = text

        image = None
        if 'image' in row:
            img_path = self.img_dir / row['image']
            try:
                image = Image.open(img_path).convert('RGB')
            except:
                image = None

        if image is None:
            image = Image.new('RGB', (224, 224), color='lightgray')

        if self.is_test or 'labels' not in row:
            labels = torch.zeros(self.num_labels)
        else:
            labels = self._encode_labels(row['labels'])

        return combined_text, image, labels, idx

def hierarchical_f1_score(y_pred_logits, y_true, ancestor_matrix, threshold=0.0):
    """محاسبه hierarchical F1 - همون کد شما"""
    y_pred = (y_pred_logits > threshold).float()
    y_true_expanded = torch.clamp(y_true @ ancestor_matrix, 0, 1)
    y_pred_expanded = torch.clamp(y_pred @ ancestor_matrix, 0, 1)

    tp = (y_true_expanded * y_pred_expanded).sum()
    true_sum = y_true_expanded.sum()
    pred_sum = y_pred_expanded.sum()

    if true_sum == 0 and pred_sum == 0:
        return 1.0, 1.0, 1.0
    elif true_sum == 0 or pred_sum == 0:
        return 0.0, 0.0, 0.0

    recall = tp / true_sum
    precision = tp / pred_sum

    if recall + precision == 0:
        f1 = 0.0
    else:
        f1 = 2 * recall * precision / (recall + precision)

    return f1.item(), recall.item(), precision.item()

# ============================================================================
# 🎯 ADABOOST IMPLEMENTATION - جدید!
# ============================================================================

class AdaBoostPropagandaDetector:
    """
    AdaBoost برای تشخیص پروپاگاندا

    مراحل:
    1. Initialize: همه samples وزن یکسان دارن
    2. Train weak learner روی weighted samples
    3. Calculate error rate
    4. Update sample weights (اشتباهات → وزن بیشتر)
    5. Calculate model weight
    6. Repeat برای n_estimators
    7. Final prediction: weighted voting
    """

    def __init__(self, config, train_df=None):
        self.config = config
        set_seed(config.seed)

        os.makedirs(config.checkpoint_dir, exist_ok=True)
        os.makedirs(config.log_dir, exist_ok=True)

        # Label mappings
        self.label_to_idx, self.idx_to_label, self.ancestor_matrix, self.num_labels = create_label_mappings()
        self.ancestor_matrix = self.ancestor_matrix.to(config.device)

        # Feature extractors (frozen)
        print("Loading feature extractors...")
        self.roberta_model = RobertaModel.from_pretrained(config.roberta_model_name).to(config.device)
        self.roberta_tokenizer = RobertaTokenizer.from_pretrained(config.roberta_model_name)
        for param in self.roberta_model.parameters():
            param.requires_grad = False
        self.roberta_model.eval()

        self.clip_model = CLIPModel.from_pretrained(config.clip_model_name).to(config.device)
        self.clip_processor = CLIPProcessor.from_pretrained(config.clip_model_name)
        for param in self.clip_model.parameters():
            param.requires_grad = False
        self.clip_model.eval()
        print("✓ Feature extractors loaded and frozen")

        # Build label graph
        if train_df is not None and config.use_gcn:
            self.adjacency_matrix = build_label_graph(
                train_df, self.label_to_idx, self.idx_to_label,
                self.roberta_model, self.roberta_tokenizer, config.device,
                config.pmi_weight, config.semantic_weight
            ).to(config.device)
        else:
            self.adjacency_matrix = None

        # AdaBoost specific
        self.weak_learners = []  # لیست مدل‌های weak
        self.model_weights = []  # وزن هر مدل
        self.sample_weights_history = []  # تاریخچه sample weights

        # Feature cache
        self.feature_cache = {}

        # Loss function
        self.focal_loss = FocalLoss(alpha=1.0, gamma=2.0)

        print(f"\n{'='*60}")
        print(f"AdaBoost Initialized:")
        print(f"  Number of weak learners: {config.n_estimators}")
        print(f"  Learning rate: {config.adaboost_learning_rate}")
        print(f"  Using GCN: {config.use_gcn}")
        print(f"{'='*60}")

    def extract_roberta_features(self, texts):
        """Extract RoBERTa features"""
        with torch.no_grad():
            encoded = self.roberta_tokenizer(
                texts, padding='max_length', truncation=True,
                max_length=256, return_tensors='pt'
            )
            input_ids = encoded['input_ids'].to(self.config.device)
            attention_mask = encoded['attention_mask'].to(self.config.device)
            outputs = self.roberta_model(input_ids=input_ids, attention_mask=attention_mask)
            return outputs.last_hidden_state[:, 0, :]

    def extract_clip_features(self, texts, images):
        """Extract CLIP features"""
        with torch.no_grad():
            inputs = self.clip_processor(
                text=texts, images=images, return_tensors='pt',
                padding='max_length', truncation=True, max_length=77
            )
            for key in inputs:
                inputs[key] = inputs[key].to(self.config.device)
            outputs = self.clip_model(**inputs)
            return torch.cat((outputs.text_embeds, outputs.image_embeds), dim=-1)

    def extract_features(self, texts, images):
        """Extract combined features"""
        roberta_features = self.extract_roberta_features(list(texts))
        clip_features = self.extract_clip_features(list(texts), list(images))
        return torch.cat((roberta_features, clip_features), dim=-1)

    def precompute_features(self, dataset, cache_name, batch_size=32):
        """Pre-compute features - همون کد شما"""
        print(f"\nPre-computing features for {cache_name}...")
        features_list = []
        labels_list = []
        num_samples = len(dataset)
        num_batches = (num_samples + batch_size - 1) // batch_size

        with torch.no_grad():
            for batch_idx in tqdm(range(num_batches), desc=f"Extracting {cache_name}"):
                start_idx = batch_idx * batch_size
                end_idx = min(start_idx + batch_size, num_samples)

                batch_texts, batch_images, batch_labels = [], [], []
                for idx in range(start_idx, end_idx):
                    text, image, label, _ = dataset[idx]
                    batch_texts.append(text)
                    batch_images.append(image)
                    batch_labels.append(label)

                features = self.extract_features(batch_texts, batch_images)
                labels_tensor = torch.stack(batch_labels)
                features_list.append(features.cpu())
                labels_list.append(labels_tensor.cpu())

        all_features = torch.cat(features_list, dim=0)
        all_labels = torch.cat(labels_list, dim=0)
        self.feature_cache[cache_name] = {'features': all_features, 'labels': all_labels}
        print(f"✓ Cached {len(all_features)} features for {cache_name}")
        return all_features, all_labels

    def create_weighted_sampler(self, dataset, sample_weights):
        """
        ایجاد WeightedRandomSampler برای sampling با وزن
        این مهم‌ترین قسمت AdaBoost هست!
        """
        sampler = WeightedRandomSampler(
            weights=sample_weights,
            num_samples=len(sample_weights),
            replacement=True
        )
        return sampler

    def train_weak_learner(self, train_loader, val_loader, learner_idx, sample_weights):
        """
        Train یک weak learner

        Args:
            train_loader: DataLoader برای training
            val_loader: DataLoader برای validation
            learner_idx: شماره weak learner
            sample_weights: وزن هر sample
        """
        print(f"\n{'='*60}")
        print(f"TRAINING WEAK LEARNER #{learner_idx + 1}")
        print(f"{'='*60}")

        # ایجاد یک مدل جدید
        model = ImprovedMultiHeadMLP(
            input_dim=1792,
            num_labels=self.num_labels,
            use_gcn=self.config.use_gcn,
            gcn_hidden_dim=self.config.gcn_hidden_dim,
            gcn_layers=self.config.gcn_layers
        ).to(self.config.device)

        # Optimizer
        optimizer = Adam(model.parameters(), lr=self.config.lr)
        scheduler = ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

        best_val_f1 = 0.0
        best_model_state = None

        # Training loop
        for epoch in range(self.config.epochs):
            model.train()
            total_loss = 0
            num_batches = 0

            # استفاده از sample weights برای محاسبه loss
            pbar = tqdm(train_loader, desc=f"Learner {learner_idx+1} - Epoch {epoch+1}")
            for batch_idx, (features, labels, indices) in enumerate(pbar):
                features = features.to(self.config.device)
                labels = labels.to(self.config.device)

                # Get sample weights برای این batch
                batch_weights = sample_weights[indices].to(self.config.device)

                optimizer.zero_grad()

                # Forward pass
                if self.adjacency_matrix is not None:
                    _, _, output = model(features, self.adjacency_matrix)
                else:
                    _, _, output = model(features)

                # Weighted focal loss
                # هر sample با وزن خودش در loss حساب میشه
                focal_loss = F.binary_cross_entropy_with_logits(output, labels, reduction='none')
                pt = torch.exp(-focal_loss)
                focal_loss = (1 - pt) ** 2.0 * focal_loss

                # Apply sample weights
                weighted_loss = (focal_loss * batch_weights.unsqueeze(1)).mean()

                loss = weighted_loss
                loss.backward()

                torch.nn.utils.clip_grad_norm_(model.parameters(), self.config.gradient_clip_norm)
                optimizer.step()

                total_loss += loss.item()
                num_batches += 1
                pbar.set_postfix({'loss': f'{loss.item():.4f}'})

            avg_loss = total_loss / num_batches

            # Validation
            val_f1 = self._validate_weak_learner(model, val_loader)
            print(f"Epoch {epoch+1} - Train Loss: {avg_loss:.4f} - Val F1: {val_f1:.4f}")

            # Save best model
            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                best_model_state = copy.deepcopy(model.state_dict())

            scheduler.step(avg_loss)

        # Load best model
        if best_model_state is not None:
            model.load_state_dict(best_model_state)

        print(f"✓ Weak Learner #{learner_idx + 1} trained - Best Val F1: {best_val_f1:.4f}")
        return model

    def _validate_weak_learner(self, model, val_loader):
        """Validate یک weak learner"""
        model.eval()
        all_logits = []
        all_labels = []

        with torch.no_grad():
            for features, labels, _ in val_loader:
                features = features.to(self.config.device)
                labels = labels.to(self.config.device)

                if self.adjacency_matrix is not None:
                    _, _, output = model(features, self.adjacency_matrix)
                else:
                    _, _, output = model(features)

                all_logits.append(output.cpu())
                all_labels.append(labels.cpu())

        all_logits = torch.cat(all_logits)
        all_labels = torch.cat(all_labels)

        f1, _, _ = hierarchical_f1_score(all_logits, all_labels, self.ancestor_matrix.cpu())
        return f1

    def calculate_model_error(self, model, train_loader, sample_weights):
        """
        محاسبه error rate یک مدل

        Error = sum(weight_i * I(prediction_i != true_i)) / sum(weights)
        """
        model.eval()
        total_weighted_error = 0.0
        total_weight = 0.0

        with torch.no_grad():
            for features, labels, indices in train_loader:
                features = features.to(self.config.device)
                labels = labels.to(self.config.device)
                batch_weights = sample_weights[indices]

                if self.adjacency_matrix is not None:
                    _, _, output = model(features, self.adjacency_matrix)
                else:
                    _, _, output = model(features)

                # Predictions
                predictions = (torch.sigmoid(output) > 0.5).float().cpu()

                # Errors per sample (any label wrong = error)
                errors = (predictions != labels.cpu()).any(dim=1).float()

                # Weighted error
                weighted_errors = errors * batch_weights
                total_weighted_error += weighted_errors.sum().item()
                total_weight += batch_weights.sum().item()

        error_rate = total_weighted_error / total_weight
        return error_rate

    def update_sample_weights(self, model, train_dataset, sample_weights):
        """
        Update sample weights بر اساس errors

        AdaBoost formula:
        - weight_new = weight_old * exp(alpha * error)
        - alpha = learning_rate * log((1 - error_rate) / error_rate)
        """
        print(f"\nUpdating sample weights...")
        model.eval()

        # Get all predictions
        all_features = self.feature_cache['train']['features'].to(self.config.device)
        all_labels = self.feature_cache['train']['labels'].to(self.config.device)

        with torch.no_grad():
            # Predict in batches
            batch_size = self.config.batch_size
            all_predictions = []

            for i in range(0, len(all_features), batch_size):
                batch_features = all_features[i:i+batch_size]

                if self.adjacency_matrix is not None:
                    _, _, output = model(batch_features, self.adjacency_matrix)
                else:
                    _, _, output = model(batch_features)

                predictions = (torch.sigmoid(output) > 0.5).float()
                all_predictions.append(predictions)

            all_predictions = torch.cat(all_predictions).cpu()

        # Calculate errors per sample
        errors = (all_predictions != all_labels.cpu()).any(dim=1).float()

        # Calculate weighted error rate
        weighted_error = (errors * sample_weights).sum() / sample_weights.sum()
        weighted_error = torch.clamp(weighted_error, min=1e-10, max=1-1e-10)

        print(f"  Weighted error rate: {weighted_error:.4f}")

        # Calculate alpha (model weight)
        alpha = self.config.adaboost_learning_rate * torch.log((1 - weighted_error) / weighted_error)
        print(f"  Model weight (alpha): {alpha:.4f}")

        # Update sample weights
        # Correct predictions: weight stays same or decreases
        # Wrong predictions: weight increases
        new_weights = sample_weights * torch.exp(alpha * errors)

        # Normalize weights
        new_weights = new_weights / new_weights.sum()
        new_weights = new_weights * len(new_weights)  # Scale back

        print(f"  Weight stats - Min: {new_weights.min():.4f}, Max: {new_weights.max():.4f}, Mean: {new_weights.mean():.4f}")
        print(f"  Samples with high weight (>2.0): {(new_weights > 2.0).sum().item()}")

        return new_weights, alpha.item()

    def fit_adaboost(self, train_dataset, val_dataset, test_dataset=None):
        """
        Main AdaBoost training loop
        """
        print(f"\n{'='*60}")
        print(f"STARTING ADABOOST TRAINING")
        print(f"{'='*60}")

        # Pre-compute features
        self.precompute_features(train_dataset, 'train', self.config.batch_size)
        self.precompute_features(val_dataset, 'val', self.config.batch_size)
        if test_dataset is not None:
            self.precompute_features(test_dataset, 'test', self.config.batch_size)

        # Initialize sample weights (uniform)
        num_samples = len(train_dataset)
        sample_weights = torch.ones(num_samples)
        print(f"\n✓ Initialized {num_samples} samples with uniform weights")

        # Training loop for each weak learner
        for i in range(self.config.n_estimators):
            print(f"\n{'#'*60}")
            print(f"ADABOOST ITERATION {i+1}/{self.config.n_estimators}")
            print(f"{'#'*60}")

            # Create weighted sampler
            weighted_sampler = self.create_weighted_sampler(train_dataset, sample_weights)

            # Create weighted train loader
            train_loader = DataLoader(
                train_dataset,
                batch_size=self.config.batch_size,
                sampler=weighted_sampler,
                collate_fn=self._collate_fn_cached,
                num_workers=0
            )

            # Regular val loader
            val_loader = DataLoader(
                val_dataset,
                batch_size=self.config.batch_size,
                shuffle=False,
                collate_fn=self._collate_fn_cached,
                num_workers=0
            )

            # Train weak learner
            weak_learner = self.train_weak_learner(train_loader, val_loader, i, sample_weights)

            # Calculate error and update weights
            if i < self.config.n_estimators - 1:  # Don't update after last learner
                sample_weights, model_weight = self.update_sample_weights(
                    weak_learner, train_dataset, sample_weights
                )
                self.sample_weights_history.append(sample_weights.clone())
            else:
                # For last model, calculate weight but don't update sample weights
                error_rate = self.calculate_model_error(weak_learner, train_loader, sample_weights)
                error_rate = max(1e-10, min(error_rate, 1-1e-10))
                model_weight = self.config.adaboost_learning_rate * np.log((1 - error_rate) / error_rate)

            # Save weak learner
            self.weak_learners.append(weak_learner)
            self.model_weights.append(model_weight)

            print(f"\n✓ Weak Learner #{i+1} added to ensemble")
            print(f"  Model weight: {model_weight:.4f}")

        # Final evaluation
        print(f"\n{'='*60}")
        print(f"ADABOOST TRAINING COMPLETED")
        print(f"{'='*60}")
        print(f"Total weak learners: {len(self.weak_learners)}")
        print(f"Model weights: {[f'{w:.3f}' for w in self.model_weights]}")

        # Evaluate ensemble
        print(f"\n--- EVALUATING ENSEMBLE ---")
        val_loader = DataLoader(
            val_dataset, batch_size=self.config.batch_size,
            shuffle=False, collate_fn=self._collate_fn_cached, num_workers=0
        )
        val_f1, val_p, val_r = self.evaluate_ensemble(val_loader, "Validation")

        if test_dataset is not None:
            test_loader = DataLoader(
                test_dataset, batch_size=self.config.batch_size,
                shuffle=False, collate_fn=self._collate_fn_cached, num_workers=0
            )
            test_f1, test_p, test_r = self.evaluate_ensemble(test_loader, "Test")

        # Save ensemble
        self.save_ensemble()

    def _collate_fn_cached(self, batch):
        """Collate function با cached features"""
        indices = [item[3] for item in batch]
        cached_data = self.feature_cache['train']
        features = cached_data['features'][indices]
        labels = cached_data['labels'][indices]
        return features.to(self.config.device), labels.to(self.config.device), torch.tensor(indices)

    def predict_ensemble(self, data_loader):
        """
        Ensemble prediction با weighted voting

        Final prediction = sum(model_weight_i * prediction_i) / sum(model_weights)
        """
        all_weighted_logits = []
        all_labels = []

        for model, weight in zip(self.weak_learners, self.model_weights):
            model.eval()
            model_logits = []

            with torch.no_grad():
                for features, labels, _ in data_loader:
                    features = features.to(self.config.device)

                    if self.adjacency_matrix is not None:
                        _, _, output = model(features, self.adjacency_matrix)
                    else:
                        _, _, output = model(features)

                    model_logits.append(output.cpu())

                    if len(all_labels) < len(labels):
                        all_labels.append(labels.cpu())

            # Weighted logits
            model_logits = torch.cat(model_logits)
            weighted_logits = model_logits * weight
            all_weighted_logits.append(weighted_logits)

        # Collect all labels
        if not all_labels:
            for features, labels, _ in data_loader:
                all_labels.append(labels.cpu())
        all_labels = torch.cat(all_labels) if len(all_labels) > 1 else all_labels[0]

        # Weighted average
        ensemble_logits = torch.stack(all_weighted_logits).sum(dim=0) / sum(self.model_weights)

        return ensemble_logits, all_labels

    def evaluate_ensemble(self, data_loader, dataset_name="Validation"):
        """Evaluate ensemble"""
        print(f"\n{'='*60}")
        print(f"EVALUATING ENSEMBLE ON {dataset_name.upper()}")
        print(f"{'='*60}")

        ensemble_logits, all_labels = self.predict_ensemble(data_loader)

        # Calculate metrics
        f1, precision, recall = hierarchical_f1_score(
            ensemble_logits, all_labels, self.ancestor_matrix.cpu()
        )

        print(f"\n{dataset_name} Results:")
        print(f"  Hierarchical F1: {f1:.4f}")
        print(f"  Hierarchical Precision: {precision:.4f}")
        print(f"  Hierarchical Recall: {recall:.4f}")

        # Per-model performance
        print(f"\n--- Individual Weak Learner Performance ---")
        for i, (model, weight) in enumerate(zip(self.weak_learners, self.model_weights)):
            model.eval()
            model_logits = []

            with torch.no_grad():
                for features, labels, _ in data_loader:
                    features = features.to(self.config.device)
                    if self.adjacency_matrix is not None:
                        _, _, output = model(features, self.adjacency_matrix)
                    else:
                        _, _, output = model(features)
                    model_logits.append(output.cpu())

            model_logits = torch.cat(model_logits)
            model_f1, _, _ = hierarchical_f1_score(model_logits, all_labels, self.ancestor_matrix.cpu())
            print(f"  Learner {i+1} (weight={weight:.3f}): F1={model_f1:.4f}")

        print(f"{'='*60}")
        return f1, precision, recall

    def save_ensemble(self):
        """Save ensemble"""
        checkpoint = {
            'config': {
                'n_estimators': self.config.n_estimators,
                'adaboost_learning_rate': self.config.adaboost_learning_rate,
                'use_gcn': self.config.use_gcn
            },
            'weak_learners': [model.state_dict() for model in self.weak_learners],
            'model_weights': self.model_weights,
            'label_to_idx': self.label_to_idx,
            'idx_to_label': self.idx_to_label,
            'ancestor_matrix': self.ancestor_matrix,
            'adjacency_matrix': self.adjacency_matrix,
            'sample_weights_history': self.sample_weights_history
        }

        save_path = os.path.join(self.config.checkpoint_dir, 'adaboost_ensemble.pth')
        torch.save(checkpoint, save_path)
        print(f"\n✓ AdaBoost ensemble saved to: {save_path}")

    def load_ensemble(self, checkpoint_path):
        """Load ensemble"""
        checkpoint = torch.load(checkpoint_path, map_location=self.config.device)

        self.model_weights = checkpoint['model_weights']
        self.label_to_idx = checkpoint['label_to_idx']
        self.idx_to_label = checkpoint['idx_to_label']
        self.ancestor_matrix = checkpoint['ancestor_matrix'].to(self.config.device)
        self.adjacency_matrix = checkpoint['adjacency_matrix']
        if self.adjacency_matrix is not None:
            self.adjacency_matrix = self.adjacency_matrix.to(self.config.device)

        # Load weak learners
        self.weak_learners = []
        for state_dict in checkpoint['weak_learners']:
            model = ImprovedMultiHeadMLP(
                input_dim=1792,
                num_labels=self.num_labels,
                use_gcn=self.config.use_gcn,
                gcn_hidden_dim=self.config.gcn_hidden_dim,
                gcn_layers=self.config.gcn_layers
            ).to(self.config.device)
            model.load_state_dict(state_dict)
            model.eval()
            self.weak_learners.append(model)

        print(f"✓ AdaBoost ensemble loaded from: {checkpoint_path}")
        print(f"  {len(self.weak_learners)} weak learners loaded")

# ============================================================================
# MAIN FUNCTION
# ============================================================================

def main():
    """Main function با AdaBoost"""
    config = CFG()
    set_seed(config.seed)

    print("="*60)
    print("ADABOOST PROPAGANDA DETECTION")
    print("="*60)

    # Load data
    with open(config.train_json) as fp:
        train = json.load(fp)
    with open(config.val_json) as fp:
        valid = json.load(fp)

    test = None
    if os.path.exists(config.test_json):
        with open(config.test_json) as fp:
            test = json.load(fp)

    train_df = pd.DataFrame(train)
    valid_df = pd.DataFrame(valid)
    test_df = pd.DataFrame(test) if test is not None else None

    print(f"✓ Train: {len(train_df)}, Val: {len(valid_df)}", end="")
    if test_df is not None:
        print(f", Test: {len(test_df)}")
    else:
        print()

    # Initialize AdaBoost detector
    detector = AdaBoostPropagandaDetector(config, train_df=train_df)

    # Create datasets
    train_dataset = MemeDataset(
        train_df, config.train_img_dir, detector.clip_processor,
        detector.label_to_idx, detector.ancestor_matrix,
        use_caption=config.use_caption, caption_separator=config.caption_separator
    )

    val_dataset = MemeDataset(
        valid_df, config.val_img_dir, detector.clip_processor,
        detector.label_to_idx, detector.ancestor_matrix,
        use_caption=config.use_caption, caption_separator=config.caption_separator
    )

    test_dataset = None
    if test_df is not None:
        test_dataset = MemeDataset(
            test_df, config.test_img_dir, detector.clip_processor,
            detector.label_to_idx, detector.ancestor_matrix,
            use_caption=config.use_caption, caption_separator=config.caption_separator
        )

    # Train AdaBoost ensemble
    detector.fit_adaboost(train_dataset, val_dataset, test_dataset)

    print("\n" + "="*60)
    print("ADABOOST TRAINING COMPLETED!")
    print("="*60)
    print("\nKey Differences from Base Model:")
    print("✓ Multiple weak learners instead of single model")
    print("✓ Sample weighting: hard examples get more attention")
    print("✓ Weighted voting: better learners have more influence")
    print("✓ Progressive learning: each model focuses on previous mistakes")
    print("="*60)

if __name__ == "__main__":
    main()

ADABOOST PROPAGANDA DETECTION
✓ Train: 7000, Val: 500, Test: 1000
Loading feature extractors...


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

Learner 5 - Epoch 7: 100%|██████████| 219/219 [01:46<00:00,  2.05it/s, loss=0.0402]


Epoch 7 - Train Loss: 0.0379 - Val F1: 0.7795


Learner 5 - Epoch 8: 100%|██████████| 219/219 [01:46<00:00,  2.05it/s, loss=0.0235]


Epoch 8 - Train Loss: 0.0361 - Val F1: 0.7873
✓ Weak Learner #5 trained - Best Val F1: 0.7873

✓ Weak Learner #5 added to ensemble
  Model weight: 0.6485

ADABOOST TRAINING COMPLETED
Total weak learners: 5
Model weights: ['-0.875', '-0.233', '-0.044', '0.003', '0.649']

--- EVALUATING ENSEMBLE ---

EVALUATING ENSEMBLE ON VALIDATION


RuntimeError: The size of tensor a (1012) must match the size of tensor b (500) at non-singleton dimension 0

In [3]:
"""
AdaBoost Implementation for Propaganda Detection
این کد AdaBoost رو به کد پایه شما اضافه می‌کنه
"""

import os
import json
import random
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, precision_recall_curve
from sklearn.metrics.pairwise import cosine_similarity
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from collections import Counter
import copy

from transformers import CLIPModel, CLIPProcessor, RobertaModel, RobertaTokenizer
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION - SAME AS YOUR BASE CODE
# ============================================================================
class CFG:
    # Paths
    train_json = '/content/drive/MyDrive/dataset_with_rationales_subtask2a_final (1).json'
    val_json = '/content/drive/MyDrive/validation_caption.json'
    test_json = '/content/drive/MyDrive/dev_processed.json'

    train_img_dir = '/content/train_images/train_images'
    val_img_dir   = '/content/validation_images/validation_images'
    test_img_dir  = '/content/dev_images/dev_images'

    # Hyperparameters
    seed = 42
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    batch_size = 32
    lr = 2e-3
    epochs = 8  # تعداد epoch برای هر weak learner کمتر می‌شه

    # AdaBoost specific
    n_estimators = 5  # تعداد weak learners
    adaboost_learning_rate = 0.8  # برای update کردن sample weights

    # Caption settings
    use_caption = True
    caption_separator = " [SEP] "

    # Model paths
    checkpoint_dir = './checkpoints_adaboost'
    log_dir = './logs_adaboost'

    # Model names
    clip_model_name = "openai/clip-vit-base-patch32"
    roberta_model_name = "roberta-base"

    # Training improvements
    gradient_clip_norm = 1.0

    # GCN settings
    use_gcn = True
    gcn_hidden_dim = 256
    gcn_layers = 2
    pmi_weight = 0.6
    semantic_weight = 0.4

def set_seed(seed=42):
    """Set random seeds for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ============================================================================
# همون HIERARCHY و DEFINITIONS از کد شما
# ============================================================================
HIERARCHY_GRAPH = {
    "Persuasion": ["Ethos", "Logos", "Pathos"],
    "Ethos": ["Ad Hominem", "Justification"],
    "Logos": ["Distraction", "Simplification"],
    "Pathos": ["Other", "Other"],
    "Ad Hominem": ["Name calling/Labeling", "Doubt", "Smears", "Reductio ad hitlerum", "Whataboutism"],
    "Justification": ["Flag-waving", "Appeal to fear/prejudice", "Bandwagon", "Slogans"],
    "Distraction": ["Misrepresentation of Someone's Position (Straw Man)", "Presenting Irrelevant Data (Red Herring)", "Whataboutism"],
    "Simplification": ["Black-and-white Fallacy/Dictatorship", "Thought-terminating cliché", "Causal Oversimplification"],
    "Other": ["Bandwagon", "Appeal to authority", "Glittering generalities (Virtue)", "Transfer", "Repetition",
             "Obfuscation, Intentional vagueness, Confusion", "Appeal to (Strong) Emotions", "Exaggeration/Minimisation",
             "Loaded Language", "Flag-waving", "Appeal to fear/prejudice", "Transfer"]
}

TECHNIQUE_DEFINITIONS = {
    "Name calling/Labeling": "Giving a person or idea a bad label to make the audience reject them without examining evidence",
    "Repetition": "Repeating the same message over and over again so that the audience will accept it",
    "Slogans": "A brief and striking phrase that contains labeling and stereotyping",
    "Appeal to fear/prejudice": "Seeking to build support by instilling anxiety and panic in the population",
    "Doubt": "Questioning the credibility of someone or something",
    "Exaggeration/Minimisation": "Either representing something in an excessive manner or making something seem less important",
    "Flag-waving": "Playing on strong national feeling to justify or promote an action",
    "Causal Oversimplification": "Assuming a single cause when there are multiple causes behind an issue",
    "Appeal to authority": "Supposing that a claim is true because a valid authority or expert on the issue said it",
    "Black-and-white Fallacy/Dictatorship": "Presenting two alternative options as the only possibilities",
    "Thought-terminating cliché": "Words or phrases that discourage critical thought and useful discussion",
    "Whataboutism": "Discredit an opponent's position by charging them with hypocrisy without refuting their argument",
    "Reductio ad hitlerum": "Comparing something/someone to Hitler or Nazism to make the argument seem invalid",
    "Bandwagon": "Attempting to persuade the target audience to join in and take the course of action because everyone else is doing so",
    "Obfuscation, Intentional vagueness, Confusion": "Using deliberately unclear words to make the message confusing",
    "Loaded Language": "Using specific words and phrases with strong emotional implications to influence the audience",
    "Glittering generalities (Virtue)": "Words associated with highly valued concepts that are used to evoke positive emotional response",
    "Misrepresentation of Someone's Position (Straw Man)": "When an opponent's proposition is substituted with a similar one",
    "Presenting Irrelevant Data (Red Herring)": "Introducing irrelevant material to the argument to distract",
    "Transfer": "Projecting positive or negative qualities of a person, entity, object to another",
    "Appeal to (Strong) Emotions": "Attempting to develop an emotional response instead of a valid or compelling argument",
    "Smears": "A direct attack on the reputation or character of a person or group",
}

# ============================================================================
# استفاده از همون توابع کمکی شما
# ============================================================================

def create_label_mappings():
    """Create label to index mappings and ancestor matrix"""
    all_labels = set(HIERARCHY_GRAPH.keys())
    for children in HIERARCHY_GRAPH.values():
        all_labels.update(children)

    label_to_idx = {label: i for i, label in enumerate(sorted(all_labels))}
    idx_to_label = {i: label for label, i in label_to_idx.items()}
    num_labels = len(label_to_idx)

    ancestors = {label_to_idx['Persuasion']: {label_to_idx['Persuasion']}}

    for parent, children in HIERARCHY_GRAPH.items():
        parent_idx = label_to_idx[parent]
        for child in children:
            child_idx = label_to_idx[child]
            ancestors[child_idx] = ancestors.get(child_idx, set()) | ancestors.get(parent_idx, set()) | {child_idx}

    ancestor_matrix = torch.zeros((num_labels, num_labels), dtype=torch.float32)
    for node, anc_set in ancestors.items():
        ancestor_matrix[node, list(anc_set)] = 1.0

    return label_to_idx, idx_to_label, ancestor_matrix, num_labels

def compute_pmi_matrix(labels_data, label_to_idx, smooth=1e-5):
    """محاسبه PMI matrix - همون کد شما"""
    num_labels = len(label_to_idx)
    co_occurrence = np.zeros((num_labels, num_labels))
    label_counts = np.zeros(num_labels)
    total_samples = len(labels_data)

    for labels in tqdm(labels_data, desc="Computing co-occurrences"):
        label_indices = [label_to_idx[label] for label in labels if label in label_to_idx]
        for idx in label_indices:
            label_counts[idx] += 1
        for i in label_indices:
            for j in label_indices:
                co_occurrence[i, j] += 1

    pmi_matrix = np.zeros((num_labels, num_labels))
    for i in range(num_labels):
        for j in range(num_labels):
            p_i = (label_counts[i] + smooth) / total_samples
            p_j = (label_counts[j] + smooth) / total_samples
            p_ij = (co_occurrence[i, j] + smooth) / total_samples
            pmi = np.log(p_ij / (p_i * p_j))
            pmi_matrix[i, j] = max(0, pmi)

    if pmi_matrix.max() > 0:
        pmi_matrix = pmi_matrix / pmi_matrix.max()

    return pmi_matrix

def compute_semantic_similarity(label_to_idx, idx_to_label, roberta_model, roberta_tokenizer, device):
    """محاسبه semantic similarity - همون کد شما"""
    num_labels = len(label_to_idx)
    similarity_matrix = np.zeros((num_labels, num_labels))
    embeddings = []

    with torch.no_grad():
        for i in tqdm(range(num_labels), desc="Extracting embeddings"):
            label_name = idx_to_label[i]
            definition = TECHNIQUE_DEFINITIONS.get(label_name, label_name)

            encoded = roberta_tokenizer(
                definition, padding='max_length', truncation=True,
                max_length=128, return_tensors='pt'
            )

            input_ids = encoded['input_ids'].to(device)
            attention_mask = encoded['attention_mask'].to(device)
            outputs = roberta_model(input_ids=input_ids, attention_mask=attention_mask)
            embedding = outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()
            embeddings.append(embedding)

    embeddings = np.array(embeddings)
    similarity_matrix = cosine_similarity(embeddings)
    similarity_matrix = (similarity_matrix + 1) / 2
    return similarity_matrix

def build_label_graph(train_df, label_to_idx, idx_to_label, roberta_model, roberta_tokenizer, device,
                      pmi_weight=0.6, semantic_weight=0.4):
    """ساخت label graph - همون کد شما"""
    print("\n" + "="*60)
    print("BUILDING LABEL GRAPH FOR GCN")
    print("="*60)

    labels_data = train_df['labels'].tolist()
    pmi_matrix = compute_pmi_matrix(labels_data, label_to_idx)
    semantic_matrix = compute_semantic_similarity(label_to_idx, idx_to_label,
                                                   roberta_model, roberta_tokenizer, device)

    adjacency_matrix = pmi_weight * pmi_matrix + semantic_weight * semantic_matrix
    np.fill_diagonal(adjacency_matrix, 1.0)

    degree = adjacency_matrix.sum(axis=1)
    degree_inv_sqrt = np.power(degree, -0.5)
    degree_inv_sqrt[np.isinf(degree_inv_sqrt)] = 0.0
    degree_matrix_inv_sqrt = np.diag(degree_inv_sqrt)
    adjacency_matrix_norm = degree_matrix_inv_sqrt @ adjacency_matrix @ degree_matrix_inv_sqrt

    print(f"✓ Label graph built: {adjacency_matrix_norm.shape}")
    return torch.FloatTensor(adjacency_matrix_norm)

# ============================================================================
# همون کلاس‌های کمکی شما (FocalLoss, GCN, MLP, etc.)
# ============================================================================

class FocalLoss(nn.Module):
    """Focal Loss"""
    def __init__(self, alpha=1.0, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

class GraphConvolutionLayer(nn.Module):
    """Graph Convolution Layer"""
    def __init__(self, in_features, out_features, bias=True):
        super(GraphConvolutionLayer, self).__init__()
        self.weight = nn.Parameter(torch.FloatTensor(in_features, out_features))
        if bias:
            self.bias = nn.Parameter(torch.FloatTensor(out_features))
        else:
            self.register_parameter('bias', None)
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.weight)
        if self.bias is not None:
            nn.init.zeros_(self.bias)

    def forward(self, input_features, adjacency_matrix):
        support = torch.matmul(input_features, self.weight)
        output = torch.einsum('ij,bjf->bif', adjacency_matrix, support)
        if self.bias is not None:
            output = output + self.bias
        return output

class LabelGCN(nn.Module):
    """GCN برای label refinement - همون کد شما"""
    def __init__(self, num_labels, hidden_dim=256, num_layers=2, dropout=0.3):
        super(LabelGCN, self).__init__()
        self.num_labels = num_labels
        self.num_layers = num_layers
        self.gcn_layers = nn.ModuleList()

        self.gcn_layers.append(GraphConvolutionLayer(1, hidden_dim))
        for _ in range(num_layers - 2):
            self.gcn_layers.append(GraphConvolutionLayer(hidden_dim, hidden_dim))
        if num_layers > 1:
            self.gcn_layers.append(GraphConvolutionLayer(hidden_dim, 1))

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.batch_norm = nn.ModuleList([nn.BatchNorm1d(hidden_dim) for _ in range(num_layers - 1)])
        self.residual_weight = nn.Parameter(torch.FloatTensor([0.5]))

    def forward(self, predictions, adjacency_matrix):
        x = predictions.unsqueeze(-1)

        for i, gcn_layer in enumerate(self.gcn_layers):
            x = gcn_layer(x, adjacency_matrix)
            if i < len(self.gcn_layers) - 1:
                x = x.transpose(1, 2)
                x = self.batch_norm[i](x)
                x = x.transpose(1, 2)
                x = self.relu(x)
                x = self.dropout(x)

        refined = x.squeeze(-1)
        alpha = torch.sigmoid(self.residual_weight)
        output = alpha * predictions + (1 - alpha) * refined
        return output

class ImprovedMultiHeadMLP(nn.Module):
    """همون Multi-Head MLP شما با GCN"""
    def __init__(self, input_dim=1792, num_labels=22, use_gcn=True, gcn_hidden_dim=256, gcn_layers=2):
        super(ImprovedMultiHeadMLP, self).__init__()
        self.use_gcn = use_gcn

        # Head 1 layers
        self.layer1 = nn.Linear(input_dim, 768)
        self.layer1_bn = nn.BatchNorm1d(768)
        self.layer2 = nn.Linear(768, 512)
        self.layer2_bn = nn.BatchNorm1d(512)
        self.head1 = nn.Linear(512, 3)
        self.head1_reducer = nn.Linear(512, 64)
        self.head1_to_head2_residual = nn.Linear(512, 128)

        # Head 2 layers
        self.layer3 = nn.Linear(512, 256)
        self.layer3_bn = nn.BatchNorm1d(256)
        self.layer4 = nn.Linear(256, 128)
        self.layer4_bn = nn.BatchNorm1d(128)
        self.head2 = nn.Linear(128, 5)
        self.head2_reducer = nn.Linear(128, 64)
        self.head2_to_final_residual = nn.Linear(128, 64)

        # Head 3 (final) layers
        self.final_layer1 = nn.Linear(128, 96)
        self.final_layer1_bn = nn.BatchNorm1d(96)
        self.final_layer2 = nn.Linear(96, 64)
        self.final_layer2_bn = nn.BatchNorm1d(64)
        self.final_residual = nn.Linear(128, 64)
        self.final_head = nn.Linear(64, num_labels)

        # GCN
        if self.use_gcn:
            self.gcn = LabelGCN(num_labels, gcn_hidden_dim, gcn_layers, dropout=0.3)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)

    def forward(self, x, adjacency_matrix=None):
        # Head 1
        x1 = self.relu(self.layer1_bn(self.layer1(x)))
        x1 = self.dropout(x1)
        x1 = self.relu(self.layer2_bn(self.layer2(x1)))
        x1 = self.dropout(x1)
        output1 = self.head1(x1)
        head1_features = self.relu(self.head1_reducer(x1))
        head1_features = self.dropout(head1_features)

        # Head 2
        x2 = self.relu(self.layer3_bn(self.layer3(x1)))
        x2 = self.dropout(x2)
        x2 = self.relu(self.layer4_bn(self.layer4(x2)))
        x2 = self.dropout(x2)
        x2 = x2 + self.head1_to_head2_residual(x1)
        output2 = self.head2(x2)
        head2_features = self.relu(self.head2_reducer(x2))
        head2_features = self.dropout(head2_features)

        # Head 3
        combined = torch.cat([head1_features, head2_features], dim=1)
        x3 = self.relu(self.final_layer1_bn(self.final_layer1(combined)))
        x3 = self.dropout(x3)
        x3 = self.relu(self.final_layer2_bn(self.final_layer2(x3)))
        x3 = self.dropout(x3)
        x3 = x3 + self.final_residual(combined)
        output_final = self.final_head(x3)

        # GCN refinement
        if self.use_gcn and adjacency_matrix is not None:
            output_final = self.gcn(output_final, adjacency_matrix)

        return output1, output2, output_final

class MemeDataset(Dataset):
    """همون Dataset شما"""
    def __init__(self, df, img_dir, processor, label_to_idx, ancestor_matrix,
                 is_test=False, use_caption=True, caption_separator=" [SEP] "):
        self.df = df.reset_index(drop=True)
        self.img_dir = Path(img_dir)
        self.processor = processor
        self.label_to_idx = label_to_idx
        self.ancestor_matrix = ancestor_matrix
        self.num_labels = len(label_to_idx)
        self.is_test = is_test
        self.use_caption = use_caption
        self.caption_separator = caption_separator

    def __len__(self):
        return len(self.df)

    def _encode_labels(self, label_list):
        if not label_list:
            return torch.zeros(self.num_labels)

        label_indices = [self.label_to_idx[label] for label in label_list if label in self.label_to_idx]
        expanded_indices = set()
        for idx in label_indices:
            ancestors = torch.where(self.ancestor_matrix[idx] == 1)[0].tolist()
            expanded_indices.update(ancestors)

        y = torch.zeros(self.num_labels)
        y[list(expanded_indices)] = 1.0
        return y

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = row['text'] if 'text' in row and pd.notna(row['text']) else ""

        if self.use_caption and 'caption' in row and pd.notna(row['caption']):
            caption = row['caption']
            combined_text = f"{text}{self.caption_separator}{caption}"
        else:
            combined_text = text

        image = None
        if 'image' in row:
            img_path = self.img_dir / row['image']
            try:
                image = Image.open(img_path).convert('RGB')
            except:
                image = None

        if image is None:
            image = Image.new('RGB', (224, 224), color='lightgray')

        if self.is_test or 'labels' not in row:
            labels = torch.zeros(self.num_labels)
        else:
            labels = self._encode_labels(row['labels'])

        return combined_text, image, labels, idx

def hierarchical_f1_score(y_pred_logits, y_true, ancestor_matrix, threshold=0.0):
    """محاسبه hierarchical F1 - همون کد شما"""
    y_pred = (y_pred_logits > threshold).float()
    y_true_expanded = torch.clamp(y_true @ ancestor_matrix, 0, 1)
    y_pred_expanded = torch.clamp(y_pred @ ancestor_matrix, 0, 1)

    tp = (y_true_expanded * y_pred_expanded).sum()
    true_sum = y_true_expanded.sum()
    pred_sum = y_pred_expanded.sum()

    if true_sum == 0 and pred_sum == 0:
        return 1.0, 1.0, 1.0
    elif true_sum == 0 or pred_sum == 0:
        return 0.0, 0.0, 0.0

    recall = tp / true_sum
    precision = tp / pred_sum

    if recall + precision == 0:
        f1 = 0.0
    else:
        f1 = 2 * recall * precision / (recall + precision)

    return f1.item(), recall.item(), precision.item()

# ============================================================================
# 🎯 ADABOOST IMPLEMENTATION - جدید!
# ============================================================================

class AdaBoostPropagandaDetector:
    """
    AdaBoost برای تشخیص پروپاگاندا

    مراحل:
    1. Initialize: همه samples وزن یکسان دارن
    2. Train weak learner روی weighted samples
    3. Calculate error rate
    4. Update sample weights (اشتباهات → وزن بیشتر)
    5. Calculate model weight
    6. Repeat برای n_estimators
    7. Final prediction: weighted voting
    """

    def __init__(self, config, train_df=None):
        self.config = config
        set_seed(config.seed)

        os.makedirs(config.checkpoint_dir, exist_ok=True)
        os.makedirs(config.log_dir, exist_ok=True)

        # Label mappings
        self.label_to_idx, self.idx_to_label, self.ancestor_matrix, self.num_labels = create_label_mappings()
        self.ancestor_matrix = self.ancestor_matrix.to(config.device)

        # Feature extractors (frozen)
        print("Loading feature extractors...")
        self.roberta_model = RobertaModel.from_pretrained(config.roberta_model_name).to(config.device)
        self.roberta_tokenizer = RobertaTokenizer.from_pretrained(config.roberta_model_name)
        for param in self.roberta_model.parameters():
            param.requires_grad = False
        self.roberta_model.eval()

        self.clip_model = CLIPModel.from_pretrained(config.clip_model_name).to(config.device)
        self.clip_processor = CLIPProcessor.from_pretrained(config.clip_model_name)
        for param in self.clip_model.parameters():
            param.requires_grad = False
        self.clip_model.eval()
        print("✓ Feature extractors loaded and frozen")

        # Build label graph
        if train_df is not None and config.use_gcn:
            self.adjacency_matrix = build_label_graph(
                train_df, self.label_to_idx, self.idx_to_label,
                self.roberta_model, self.roberta_tokenizer, config.device,
                config.pmi_weight, config.semantic_weight
            ).to(config.device)
        else:
            self.adjacency_matrix = None

        # AdaBoost specific
        self.weak_learners = []  # لیست مدل‌های weak
        self.model_weights = []  # وزن هر مدل
        self.sample_weights_history = []  # تاریخچه sample weights

        # Feature cache
        self.feature_cache = {}

        # Loss function
        self.focal_loss = FocalLoss(alpha=1.0, gamma=2.0)

        print(f"\n{'='*60}")
        print(f"AdaBoost Initialized:")
        print(f"  Number of weak learners: {config.n_estimators}")
        print(f"  Learning rate: {config.adaboost_learning_rate}")
        print(f"  Using GCN: {config.use_gcn}")
        print(f"{'='*60}")

    def extract_roberta_features(self, texts):
        """Extract RoBERTa features"""
        with torch.no_grad():
            encoded = self.roberta_tokenizer(
                texts, padding='max_length', truncation=True,
                max_length=256, return_tensors='pt'
            )
            input_ids = encoded['input_ids'].to(self.config.device)
            attention_mask = encoded['attention_mask'].to(self.config.device)
            outputs = self.roberta_model(input_ids=input_ids, attention_mask=attention_mask)
            return outputs.last_hidden_state[:, 0, :]

    def extract_clip_features(self, texts, images):
        """Extract CLIP features"""
        with torch.no_grad():
            inputs = self.clip_processor(
                text=texts, images=images, return_tensors='pt',
                padding='max_length', truncation=True, max_length=77
            )
            for key in inputs:
                inputs[key] = inputs[key].to(self.config.device)
            outputs = self.clip_model(**inputs)
            return torch.cat((outputs.text_embeds, outputs.image_embeds), dim=-1)

    def extract_features(self, texts, images):
        """Extract combined features"""
        roberta_features = self.extract_roberta_features(list(texts))
        clip_features = self.extract_clip_features(list(texts), list(images))
        return torch.cat((roberta_features, clip_features), dim=-1)

    def precompute_features(self, dataset, cache_name, batch_size=32):
        """Pre-compute features - همون کد شما"""
        print(f"\nPre-computing features for {cache_name}...")
        features_list = []
        labels_list = []
        num_samples = len(dataset)
        num_batches = (num_samples + batch_size - 1) // batch_size

        with torch.no_grad():
            for batch_idx in tqdm(range(num_batches), desc=f"Extracting {cache_name}"):
                start_idx = batch_idx * batch_size
                end_idx = min(start_idx + batch_size, num_samples)

                batch_texts, batch_images, batch_labels = [], [], []
                for idx in range(start_idx, end_idx):
                    text, image, label, _ = dataset[idx]
                    batch_texts.append(text)
                    batch_images.append(image)
                    batch_labels.append(label)

                features = self.extract_features(batch_texts, batch_images)
                labels_tensor = torch.stack(batch_labels)
                features_list.append(features.cpu())
                labels_list.append(labels_tensor.cpu())

        all_features = torch.cat(features_list, dim=0)
        all_labels = torch.cat(labels_list, dim=0)
        self.feature_cache[cache_name] = {'features': all_features, 'labels': all_labels}
        print(f"✓ Cached {len(all_features)} features for {cache_name}")
        return all_features, all_labels

    def create_weighted_sampler(self, dataset, sample_weights):
        """
        ایجاد WeightedRandomSampler برای sampling با وزن
        این مهم‌ترین قسمت AdaBoost هست!
        """
        sampler = WeightedRandomSampler(
            weights=sample_weights,
            num_samples=len(sample_weights),
            replacement=True
        )
        return sampler

    def train_weak_learner(self, train_loader, val_loader, learner_idx, sample_weights):
        """
        Train یک weak learner

        Args:
            train_loader: DataLoader برای training
            val_loader: DataLoader برای validation
            learner_idx: شماره weak learner
            sample_weights: وزن هر sample
        """
        print(f"\n{'='*60}")
        print(f"TRAINING WEAK LEARNER #{learner_idx + 1}")
        print(f"{'='*60}")

        # ایجاد یک مدل جدید
        model = ImprovedMultiHeadMLP(
            input_dim=1792,
            num_labels=self.num_labels,
            use_gcn=self.config.use_gcn,
            gcn_hidden_dim=self.config.gcn_hidden_dim,
            gcn_layers=self.config.gcn_layers
        ).to(self.config.device)

        # Optimizer
        optimizer = Adam(model.parameters(), lr=self.config.lr)
        scheduler = ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

        best_val_f1 = 0.0
        best_model_state = None

        # Training loop
        for epoch in range(self.config.epochs):
            model.train()
            total_loss = 0
            num_batches = 0

            # استفاده از sample weights برای محاسبه loss
            pbar = tqdm(train_loader, desc=f"Learner {learner_idx+1} - Epoch {epoch+1}")
            for batch_idx, (features, labels, indices) in enumerate(pbar):
                features = features.to(self.config.device)
                labels = labels.to(self.config.device)

                # Get sample weights برای این batch
                batch_weights = sample_weights[indices].to(self.config.device)

                optimizer.zero_grad()

                # Forward pass
                if self.adjacency_matrix is not None:
                    _, _, output = model(features, self.adjacency_matrix)
                else:
                    _, _, output = model(features)

                # Weighted focal loss
                # هر sample با وزن خودش در loss حساب میشه
                focal_loss = F.binary_cross_entropy_with_logits(output, labels, reduction='none')
                pt = torch.exp(-focal_loss)
                focal_loss = (1 - pt) ** 2.0 * focal_loss

                # Apply sample weights
                weighted_loss = (focal_loss * batch_weights.unsqueeze(1)).mean()

                loss = weighted_loss
                loss.backward()

                torch.nn.utils.clip_grad_norm_(model.parameters(), self.config.gradient_clip_norm)
                optimizer.step()

                total_loss += loss.item()
                num_batches += 1
                pbar.set_postfix({'loss': f'{loss.item():.4f}'})

            avg_loss = total_loss / num_batches

            # Validation
            val_f1 = self._validate_weak_learner(model, val_loader)
            print(f"Epoch {epoch+1} - Train Loss: {avg_loss:.4f} - Val F1: {val_f1:.4f}")

            # Save best model
            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                best_model_state = copy.deepcopy(model.state_dict())

            scheduler.step(avg_loss)

        # Load best model
        if best_model_state is not None:
            model.load_state_dict(best_model_state)

        print(f"✓ Weak Learner #{learner_idx + 1} trained - Best Val F1: {best_val_f1:.4f}")
        return model

    def _validate_weak_learner(self, model, val_loader):
        """Validate یک weak learner"""
        model.eval()
        all_logits = []
        all_labels = []

        with torch.no_grad():
            for features, labels, _ in val_loader:
                features = features.to(self.config.device)
                labels = labels.to(self.config.device)

                if self.adjacency_matrix is not None:
                    _, _, output = model(features, self.adjacency_matrix)
                else:
                    _, _, output = model(features)

                all_logits.append(output.cpu())
                all_labels.append(labels.cpu())

        all_logits = torch.cat(all_logits)
        all_labels = torch.cat(all_labels)

        f1, _, _ = hierarchical_f1_score(all_logits, all_labels, self.ancestor_matrix.cpu())
        return f1

    def calculate_model_error(self, model, train_loader, sample_weights):
        """
        محاسبه error rate یک مدل

        Error = sum(weight_i * I(prediction_i != true_i)) / sum(weights)
        """
        model.eval()
        total_weighted_error = 0.0
        total_weight = 0.0

        with torch.no_grad():
            for features, labels, indices in train_loader:
                features = features.to(self.config.device)
                labels = labels.to(self.config.device)
                batch_weights = sample_weights[indices]

                if self.adjacency_matrix is not None:
                    _, _, output = model(features, self.adjacency_matrix)
                else:
                    _, _, output = model(features)

                # Predictions
                predictions = (torch.sigmoid(output) > 0.5).float().cpu()

                # Errors per sample (any label wrong = error)
                errors = (predictions != labels.cpu()).any(dim=1).float()

                # Weighted error
                weighted_errors = errors * batch_weights
                total_weighted_error += weighted_errors.sum().item()
                total_weight += batch_weights.sum().item()

        error_rate = total_weighted_error / total_weight
        return error_rate

    def update_sample_weights(self, model, train_dataset, sample_weights):
        """
        Update sample weights بر اساس errors

        AdaBoost formula:
        - weight_new = weight_old * exp(alpha * error)
        - alpha = learning_rate * log((1 - error_rate) / error_rate)
        """
        print(f"\nUpdating sample weights...")
        model.eval()

        # Get all predictions
        all_features = self.feature_cache['train']['features'].to(self.config.device)
        all_labels = self.feature_cache['train']['labels'].to(self.config.device)

        with torch.no_grad():
            # Predict in batches
            batch_size = self.config.batch_size
            all_predictions = []

            for i in range(0, len(all_features), batch_size):
                batch_features = all_features[i:i+batch_size]

                if self.adjacency_matrix is not None:
                    _, _, output = model(batch_features, self.adjacency_matrix)
                else:
                    _, _, output = model(batch_features)

                predictions = (torch.sigmoid(output) > 0.5).float()
                all_predictions.append(predictions)

            all_predictions = torch.cat(all_predictions).cpu()

        # Calculate errors per sample
        errors = (all_predictions != all_labels.cpu()).any(dim=1).float()

        # Calculate weighted error rate
        weighted_error = (errors * sample_weights).sum() / sample_weights.sum()
        weighted_error = torch.clamp(weighted_error, min=1e-10, max=1-1e-10)

        print(f"  Weighted error rate: {weighted_error:.4f}")

        # Calculate alpha (model weight)
        alpha = self.config.adaboost_learning_rate * torch.log((1 - weighted_error) / weighted_error)
        print(f"  Model weight (alpha): {alpha:.4f}")

        # Update sample weights
        # Correct predictions: weight stays same or decreases
        # Wrong predictions: weight increases
        new_weights = sample_weights * torch.exp(alpha * errors)

        # Normalize weights
        new_weights = new_weights / new_weights.sum()
        new_weights = new_weights * len(new_weights)  # Scale back

        print(f"  Weight stats - Min: {new_weights.min():.4f}, Max: {new_weights.max():.4f}, Mean: {new_weights.mean():.4f}")
        print(f"  Samples with high weight (>2.0): {(new_weights > 2.0).sum().item()}")

        return new_weights, alpha.item()

    def fit_adaboost(self, train_dataset, val_dataset, test_dataset=None):
        """
        Main AdaBoost training loop
        """
        print(f"\n{'='*60}")
        print(f"STARTING ADABOOST TRAINING")
        print(f"{'='*60}")

        # Pre-compute features
        self.precompute_features(train_dataset, 'train', self.config.batch_size)
        self.precompute_features(val_dataset, 'val', self.config.batch_size)
        if test_dataset is not None:
            self.precompute_features(test_dataset, 'test', self.config.batch_size)

        # Initialize sample weights (uniform)
        num_samples = len(train_dataset)
        sample_weights = torch.ones(num_samples)
        print(f"\n✓ Initialized {num_samples} samples with uniform weights")

        # Training loop for each weak learner
        for i in range(self.config.n_estimators):
            print(f"\n{'#'*60}")
            print(f"ADABOOST ITERATION {i+1}/{self.config.n_estimators}")
            print(f"{'#'*60}")

            # Create weighted sampler
            weighted_sampler = self.create_weighted_sampler(train_dataset, sample_weights)

            # Create weighted train loader
            train_loader = DataLoader(
                train_dataset,
                batch_size=self.config.batch_size,
                sampler=weighted_sampler,
                collate_fn=self._collate_fn_cached,
                num_workers=0
            )

            # Regular val loader
            val_loader = DataLoader(
                val_dataset,
                batch_size=self.config.batch_size,
                shuffle=False,
                collate_fn=self._collate_fn_cached,
                num_workers=0
            )

            # Train weak learner
            weak_learner = self.train_weak_learner(train_loader, val_loader, i, sample_weights)

            # Calculate error and update weights
            if i < self.config.n_estimators - 1:  # Don't update after last learner
                sample_weights, model_weight = self.update_sample_weights(
                    weak_learner, train_dataset, sample_weights
                )
                self.sample_weights_history.append(sample_weights.clone())
            else:
                # For last model, calculate weight but don't update sample weights
                error_rate = self.calculate_model_error(weak_learner, train_loader, sample_weights)
                error_rate = max(1e-10, min(error_rate, 1-1e-10))
                model_weight = self.config.adaboost_learning_rate * np.log((1 - error_rate) / error_rate)

            # Save weak learner
            self.weak_learners.append(weak_learner)
            self.model_weights.append(model_weight)

            print(f"\n✓ Weak Learner #{i+1} added to ensemble")
            print(f"  Model weight: {model_weight:.4f}")

        # Final evaluation
        print(f"\n{'='*60}")
        print(f"ADABOOST TRAINING COMPLETED")
        print(f"{'='*60}")
        print(f"Total weak learners: {len(self.weak_learners)}")
        print(f"Model weights: {[f'{w:.3f}' for w in self.model_weights]}")

        # Evaluate ensemble
        print(f"\n--- EVALUATING ENSEMBLE ---")
        val_loader = DataLoader(
            val_dataset, batch_size=self.config.batch_size,
            shuffle=False, collate_fn=self._collate_fn_cached, num_workers=0
        )
        val_f1, val_p, val_r = self.evaluate_ensemble(val_loader, "Validation")

        if test_dataset is not None:
            test_loader = DataLoader(
                test_dataset, batch_size=self.config.batch_size,
                shuffle=False, collate_fn=self._collate_fn_cached, num_workers=0
            )
            test_f1, test_p, test_r = self.evaluate_ensemble(test_loader, "Test")

        # Save ensemble
        self.save_ensemble()

    def _collate_fn_cached(self, batch):
        """Collate function با cached features"""
        indices = [item[3] for item in batch]
        cached_data = self.feature_cache['train']
        features = cached_data['features'][indices]
        labels = cached_data['labels'][indices]
        return features.to(self.config.device), labels.to(self.config.device), torch.tensor(indices)

    def predict_ensemble(self, data_loader):
        """
        Ensemble prediction با weighted voting

        Final prediction = sum(model_weight_i * prediction_i) / sum(model_weights)
        """
        all_weighted_logits = []
        all_labels = []
        labels_collected = False

        for model, weight in zip(self.weak_learners, self.model_weights):
            model.eval()
            model_logits = []

            with torch.no_grad():
                for features, labels, _ in data_loader:
                    features = features.to(self.config.device)

                    if self.adjacency_matrix is not None:
                        _, _, output = model(features, self.adjacency_matrix)
                    else:
                        _, _, output = model(features)

                    model_logits.append(output.cpu())

                    # Collect labels only once (first model, all batches)
                    if not labels_collected:
                        all_labels.append(labels.cpu())

            # Mark labels as collected after first model
            labels_collected = True

            # Weighted logits
            model_logits = torch.cat(model_logits)
            weighted_logits = model_logits * weight
            all_weighted_logits.append(weighted_logits)

        # Concatenate all labels
        all_labels = torch.cat(all_labels)

        # Weighted average
        ensemble_logits = torch.stack(all_weighted_logits).sum(dim=0) / sum(self.model_weights)

        return ensemble_logits, all_labels

    def evaluate_ensemble(self, data_loader, dataset_name="Validation"):
        """Evaluate ensemble"""
        print(f"\n{'='*60}")
        print(f"EVALUATING ENSEMBLE ON {dataset_name.upper()}")
        print(f"{'='*60}")

        ensemble_logits, all_labels = self.predict_ensemble(data_loader)

        # Calculate metrics
        f1, precision, recall = hierarchical_f1_score(
            ensemble_logits, all_labels, self.ancestor_matrix.cpu()
        )

        print(f"\n{dataset_name} Results:")
        print(f"  Hierarchical F1: {f1:.4f}")
        print(f"  Hierarchical Precision: {precision:.4f}")
        print(f"  Hierarchical Recall: {recall:.4f}")

        # Per-model performance
        print(f"\n--- Individual Weak Learner Performance ---")
        for i, (model, weight) in enumerate(zip(self.weak_learners, self.model_weights)):
            model.eval()
            model_logits = []

            with torch.no_grad():
                for features, labels, _ in data_loader:
                    features = features.to(self.config.device)
                    if self.adjacency_matrix is not None:
                        _, _, output = model(features, self.adjacency_matrix)
                    else:
                        _, _, output = model(features)
                    model_logits.append(output.cpu())

            model_logits = torch.cat(model_logits)
            model_f1, _, _ = hierarchical_f1_score(model_logits, all_labels, self.ancestor_matrix.cpu())
            print(f"  Learner {i+1} (weight={weight:.3f}): F1={model_f1:.4f}")

        print(f"{'='*60}")
        return f1, precision, recall

    def save_ensemble(self):
        """Save ensemble"""
        checkpoint = {
            'config': {
                'n_estimators': self.config.n_estimators,
                'adaboost_learning_rate': self.config.adaboost_learning_rate,
                'use_gcn': self.config.use_gcn
            },
            'weak_learners': [model.state_dict() for model in self.weak_learners],
            'model_weights': self.model_weights,
            'label_to_idx': self.label_to_idx,
            'idx_to_label': self.idx_to_label,
            'ancestor_matrix': self.ancestor_matrix,
            'adjacency_matrix': self.adjacency_matrix,
            'sample_weights_history': self.sample_weights_history
        }

        save_path = os.path.join(self.config.checkpoint_dir, 'adaboost_ensemble.pth')
        torch.save(checkpoint, save_path)
        print(f"\n✓ AdaBoost ensemble saved to: {save_path}")

    def load_ensemble(self, checkpoint_path):
        """Load ensemble"""
        checkpoint = torch.load(checkpoint_path, map_location=self.config.device)

        self.model_weights = checkpoint['model_weights']
        self.label_to_idx = checkpoint['label_to_idx']
        self.idx_to_label = checkpoint['idx_to_label']
        self.ancestor_matrix = checkpoint['ancestor_matrix'].to(self.config.device)
        self.adjacency_matrix = checkpoint['adjacency_matrix']
        if self.adjacency_matrix is not None:
            self.adjacency_matrix = self.adjacency_matrix.to(self.config.device)

        # Load weak learners
        self.weak_learners = []
        for state_dict in checkpoint['weak_learners']:
            model = ImprovedMultiHeadMLP(
                input_dim=1792,
                num_labels=self.num_labels,
                use_gcn=self.config.use_gcn,
                gcn_hidden_dim=self.config.gcn_hidden_dim,
                gcn_layers=self.config.gcn_layers
            ).to(self.config.device)
            model.load_state_dict(state_dict)
            model.eval()
            self.weak_learners.append(model)

        print(f"✓ AdaBoost ensemble loaded from: {checkpoint_path}")
        print(f"  {len(self.weak_learners)} weak learners loaded")

# ============================================================================
# MAIN FUNCTION
# ============================================================================

def main():
    """Main function با AdaBoost"""
    config = CFG()
    set_seed(config.seed)

    print("="*60)
    print("ADABOOST PROPAGANDA DETECTION")
    print("="*60)

    # Load data
    with open(config.train_json) as fp:
        train = json.load(fp)
    with open(config.val_json) as fp:
        valid = json.load(fp)

    test = None
    if os.path.exists(config.test_json):
        with open(config.test_json) as fp:
            test = json.load(fp)

    train_df = pd.DataFrame(train)
    valid_df = pd.DataFrame(valid)
    test_df = pd.DataFrame(test) if test is not None else None

    print(f"✓ Train: {len(train_df)}, Val: {len(valid_df)}", end="")
    if test_df is not None:
        print(f", Test: {len(test_df)}")
    else:
        print()

    # Initialize AdaBoost detector
    detector = AdaBoostPropagandaDetector(config, train_df=train_df)

    # Create datasets
    train_dataset = MemeDataset(
        train_df, config.train_img_dir, detector.clip_processor,
        detector.label_to_idx, detector.ancestor_matrix,
        use_caption=config.use_caption, caption_separator=config.caption_separator
    )

    val_dataset = MemeDataset(
        valid_df, config.val_img_dir, detector.clip_processor,
        detector.label_to_idx, detector.ancestor_matrix,
        use_caption=config.use_caption, caption_separator=config.caption_separator
    )

    test_dataset = None
    if test_df is not None:
        test_dataset = MemeDataset(
            test_df, config.test_img_dir, detector.clip_processor,
            detector.label_to_idx, detector.ancestor_matrix,
            use_caption=config.use_caption, caption_separator=config.caption_separator
        )

    # Train AdaBoost ensemble
    detector.fit_adaboost(train_dataset, val_dataset, test_dataset)

    print("\n" + "="*60)
    print("ADABOOST TRAINING COMPLETED!")
    print("="*60)
    print("\nKey Differences from Base Model:")
    print("✓ Multiple weak learners instead of single model")
    print("✓ Sample weighting: hard examples get more attention")
    print("✓ Weighted voting: better learners have more influence")
    print("✓ Progressive learning: each model focuses on previous mistakes")
    print("="*60)

if __name__ == "__main__":
    main()

ADABOOST PROPAGANDA DETECTION
✓ Train: 7000, Val: 500, Test: 1000
Loading feature extractors...


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

Epoch 6 - Train Loss: 0.0398 - Val F1: 0.7852


Learner 5 - Epoch 7: 100%|██████████| 219/219 [01:56<00:00,  1.87it/s, loss=0.0353]


Epoch 7 - Train Loss: 0.0387 - Val F1: 0.7750


Learner 5 - Epoch 8: 100%|██████████| 219/219 [01:59<00:00,  1.83it/s, loss=0.0330]


Epoch 8 - Train Loss: 0.0373 - Val F1: 0.7680
✓ Weak Learner #5 trained - Best Val F1: 0.7852

✓ Weak Learner #5 added to ensemble
  Model weight: 0.1101

ADABOOST TRAINING COMPLETED
Total weak learners: 5
Model weights: ['-0.860', '-0.312', '-0.050', '0.089', '0.110']

--- EVALUATING ENSEMBLE ---

EVALUATING ENSEMBLE ON VALIDATION

Validation Results:
  Hierarchical F1: 0.8410
  Hierarchical Precision: 0.7949
  Hierarchical Recall: 0.8928

--- Individual Weak Learner Performance ---
  Learner 1 (weight=-0.860): F1=0.8441
  Learner 2 (weight=-0.312): F1=0.8054
  Learner 3 (weight=-0.050): F1=0.8038
  Learner 4 (weight=0.089): F1=0.7980
  Learner 5 (weight=0.110): F1=0.7852

EVALUATING ENSEMBLE ON TEST

Test Results:
  Hierarchical F1: 0.8332
  Hierarchical Precision: 0.7850
  Hierarchical Recall: 0.8877

--- Individual Weak Learner Performance ---
  Learner 1 (weight=-0.860): F1=0.8347
  Learner 2 (weight=-0.312): F1=0.7962
  Learner 3 (weight=-0.050): F1=0.7955
  Learner 4 (weight=0.0